In [93]:
from z3 import *

field=[
    [0,0,2,-1],
    [1,1,2,-1],
    [-1,-1,-1,-1],
    [-1,-1,-1,-1]
]
dim=[4,4]

s=Solver()
#TODO use array to describe mines and find box to open faster than variable based solution(in which case we need to iterate troughh unknnown boxes and vast time)
mines = Function("mines", IntSort(),IntSort(),BoolSort())
fx,fy=Ints("fx fy")

def is_correct_mineboard(mines):
    conds = []
    conds.append(ForAll([fx, fy], Implies(Not(And(0<=fx,fx<dim[0],0<=fy,fy<dim[1])),   Not(mines[fx,fy]))))

    def append(a,m,x,y):
        if 0<=x and x<dim[0] and 0<=y and y<dim[1]:
            a.append(m(x,y))#a.append(m[x,y])

    for x in range(dim[0]):
        for y in range(dim[1]):
            if field[x][y]>=0:
                next_pieces = []
                append(next_pieces,mines,x-1,y)
                append(next_pieces,mines,x+1,y)
                append(next_pieces,mines,x-1,y-1)
                append(next_pieces,mines,x,y-1)
                append(next_pieces,mines,x+1,y-1)
                append(next_pieces,mines,x-1,y+1)
                append(next_pieces,mines,x,y+1)
                append(next_pieces,mines,x+1,y+1)
                conds.append(Sum([next_piece for next_piece in next_pieces])==field[x][y])
    return And(conds)

second_mines = Function("mines", IntSort(),IntSort(),BoolSort())

x,y = Ints("x y")
s.add(And(0<=x,x<dim[0],0<=y,y<dim[1],
          is_correct_mineboard(mines),
          Not(Exists([second_mines],And(is_correct_mineboard(second_mines),second_mines[x,y]!=minesx,y])))))


while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m.eval(x),m.eval(y))
        break
    else:
        print(st)
        break


Z3Exception: Invalid bounded variable(s)

In [1]:
from z3 import *
from pprint import pprint
import json
import sys

# Mines
# https://www.chiark.greenend.org.uk/~sgtatham/puzzles/js/mines.html

U=-2# unknown
M=-1# mine 
O=-3# should be opened

def to_str(s):
    if s==U:
        return ""
    elif s==M:
        return "M"
    elif s==O:
        return "X"
    else:
        return str(s)

def to_int(s):
    if s=="":
        return U
    elif s=="M":
        return M
    elif s=="X":
        return O
    else:
        return int(s) 

#dim=[24,24]
dim=[9,9]
#number_of_mines=99
number_of_mines=30
initial =[[0, 1, -1, -1, -1, -1, 2, -2, -2], [0, 2, 4, -1, 4, 2, 2, 4, -2], [2, 3, -1, 2, 1, 0, 0, 3, -2], [-1, -1, 2, 1, 1, 2, 2, 3, -2], [-2, -2, -2, -2, -2, -2, -2, -2, -2], [-2, -2, -2, -2, -2, -2, -2, -2, -2], [-2, -2, -2, -2, -2, -2, -2, -2, -2], [-2, -2, -2, -2, -2, -2, -2, -2, -2], [-2, -2, -2, -2, -2, -2, -2, -2, -2]] 
#initial=[]

import ipywidgets as widgets
from IPython.display import display, clear_output

def create_matrix_widget(rows, cols):
    return [[widgets.Text(value='', description=f'',style={'description_width': 'initial'},layout = widgets.Layout(width='30px')) for j in range(cols)] for i in range(rows)]



def get_matrix_values(matrix_widget):
    return [[to_int(cell.value) for cell in row] for row in matrix_widget]

def set_matrix_values(matrix_widget,field):
    for x in range(dim[0]):
        for y in range(dim[1]):
            matrix_widget[x][y].value = to_str(field[x][y])
    return matrix_widget


#TODO first we should check neightboroing boxes of openedd boxes
def find_next(field,num_of_mines):
    mines=[[Const("mines_%i_%i" %(i,j),BoolSort()) for i in range(dim[1])] for j in range(dim[0])]

    def is_correct_mineboard(mines, field):
        conds = []
        conds.append(Sum([var for row in second_mines for var in row] )==num_of_mines)

        def append(a,m,x,y):
            if 0<=x and x<dim[0] and 0<=y and y<dim[1]:
                a.append(m[x][y])#a.append(m[x,y])

        for x in range(dim[0]):
            for y in range(dim[1]):
                if field[x][y]>=0:
                    next_pieces = []
                    append(next_pieces,mines,x-1,y)
                    append(next_pieces,mines,x+1,y)
                    append(next_pieces,mines,x-1,y-1)
                    append(next_pieces,mines,x,y-1)
                    append(next_pieces,mines,x+1,y-1)
                    append(next_pieces,mines,x-1,y+1)
                    append(next_pieces,mines,x,y+1)
                    append(next_pieces,mines,x+1,y+1)
                    conds.append(Sum([next_piece for next_piece in next_pieces])==field[x][y])
                    conds.append(Not(mines[x][y]))
                elif field[x][y]==M:
                    conds.append(mines[x][y])
        return And(conds)

    second_mines=[[Const("second_mines_%i_%i" %(i,j),BoolSort()) for i in range(dim[1])] for j in range(dim[0])]
    second_mines_vars = [var for row in second_mines for var in row] 
    while True:
        
        for x in range(dim[0]):
            for y in range(dim[1]):
                
                if field[x][y]!=U:
                    continue
                print("checked ",x,y)
                s = Solver()
                s.add(And(
                    is_correct_mineboard(mines,field),
                    Not(Exists(second_mines_vars,And(is_correct_mineboard(second_mines,field),
                                                second_mines[x][y]!=mines[x][y])))))
                st=s.check()
                
                if st==sat:
                    m = s.model()
                    res=m.eval(mines[x][y])
                    if res==True:
                        field[x][y]=M
                        continue
                    else:
                        field[x][y]=O
                        return field
        return field
matrix_widget=None

def clicked(arg):
    global matrix_widget

    
    if arg!=0:
        matrix_values = get_matrix_values(matrix_widget)
        matrix_values = find_next(matrix_values,number_of_mines)
        matrix_widget = set_matrix_values(matrix_widget,matrix_values)
    else:
        matrix_widget = create_matrix_widget(dim[0], dim[1])
        if initial:
            matrix_widget = set_matrix_values(matrix_widget,initial)
    clear_output()
    
    matrix_box = widgets.VBox([widgets.HBox(row) for row in matrix_widget])

    display(matrix_box)

    button_download = widgets.Button(description = 'Test Button')   
    button_download.on_click(clicked)
    display(button_download)


clicked(0)




Button(description='Test Button', style=ButtonStyle())

NameError: name 'number_of_mines' is not defined